# Image-adaptive 3D LUT — FiveK reproduction

Settings: **GPU T4**, Internet **on**. Add the notebook secret `KAGGLE_API_TOKEN` (kaggle.com → Settings → API → Generate New Token); the data cell downloads FiveK 480p to local disk.

To resume after a session ends: save this notebook's output as a dataset version, attach it as an input, and set `RESUME_FROM`.

In [ ]:
REPO = "https://github.com/siddharthrepo/Enhancer.git"
!rm -rf /kaggle/working/3d-lut && git clone -q $REPO /kaggle/working/3d-lut
%cd /kaggle/working/3d-lut

In [ ]:
# Download FiveK 480p via the Kaggle API to local disk (the dataset mount is unreliable).
# Needs notebook secret (Add-ons -> Secrets): KAGGLE_API_TOKEN (kaggle.com -> Settings -> API -> Generate New Token).
import os, subprocess

def find_fivek(base):
    for root, dirs, files in os.walk(base, followlinks=True):
        if 'test.txt' in files and os.path.isdir(os.path.join(root, 'input', 'JPG', '480p')):
            return root
    return None

LOCAL = '/tmp/fivek'
DATA = find_fivek('/kaggle/input') or find_fivek(LOCAL)
if DATA is None:
    from kaggle_secrets import UserSecretsClient
    os.environ['KAGGLE_API_TOKEN'] = UserSecretsClient().get_secret('KAGGLE_API_TOKEN')
    subprocess.run(['pip', 'install', '-q', '-U', 'kaggle'], check=True)  # API tokens need a recent CLI
    os.makedirs(LOCAL, exist_ok=True)
    zip_path = f'{LOCAL}/zeng-3lut-fivek-480p.zip'
    subprocess.run(['kaggle', 'datasets', 'download', '-d', 'jesucristo/zeng-3lut-fivek-480p', '-p', LOCAL], check=True)
    # only the 8-bit sRGB inputs, expert C targets and split files (~3.5 GB of the 14 GB)
    subprocess.run(['unzip', '-q', '-o', zip_path, 'fiveK480p/fiveK/*.txt', 'fiveK480p/fiveK/input/JPG/480p/*',
                    'fiveK480p/fiveK/expertC/JPG/480p/*', '-d', LOCAL], check=True)
    os.remove(zip_path)
    DATA = find_fivek(LOCAL)
assert DATA, 'FiveK still not found'
print(DATA)
for f in ('train_input.txt', 'train_label.txt', 'test.txt'):
    print(f, sum(1 for l in open(os.path.join(DATA, f)) if l.strip()))
print('inputs', len(os.listdir(f'{DATA}/input/JPG/480p')), 'targets', len(os.listdir(f'{DATA}/expertC/JPG/480p')))

In [ ]:
# sanity check: one input / expert C pair
from PIL import Image
import matplotlib.pyplot as plt
name = open(f'{DATA}/test.txt').readline().strip()
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for a, sub in zip(ax, ('input', 'expertC')):
    a.imshow(Image.open(f'{DATA}/{sub}/JPG/480p/{name}.jpg')); a.set_title(f'{sub} {name}'); a.axis('off')

In [ ]:
!python -m pytest -q

In [ ]:
RUN = '/kaggle/working/runs/fivek'
RESUME_FROM = None  # e.g. '/kaggle/input/<your-output-dataset>/runs/fivek'
if RESUME_FROM:
    os.makedirs(RUN, exist_ok=True)
    !cp {RESUME_FROM}/*.pt {RESUME_FROM}/log.jsonl {RUN}/

In [ ]:
# batch 1, lr 1e-4, 3 x 33^3 LUTs. Test PSNR every epoch (fast, on GPU); full SSIM/dE + best.pt on each new best.
# Test PSNR peaks around epoch 30 and then overfits, so 80 epochs is enough (~2.5h).
!python train.py --data {DATA} --out {RUN} --epochs 80 --batch-size 1 --workers 4 --eval-every 1 --max-hours 11

In [ ]:
!python eval.py --data {DATA} --ckpt {RUN}/best.pt --save-images

In [ ]:
# baselines + per-image oracle LUT (upper bound for a single global LUT); ~10 min on GPU
!python baselines.py --data {DATA} --out /kaggle/working/runs/baselines